<a href="https://colab.research.google.com/github/Aeman-Imtiaz/Parallel-and-Distributed-Computing/blob/main/C_Processes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%bash
cat <<EOF > code.c
#include <stdio.h>
#include <unistd.h>
#include <string.h>

int main() {

    int fd[2];  // fd[0] = read, fd[1] = write

    pipe(fd);

    int id = fork();

    if(id > 0) {
        // Parent process

        close(fd[0]); // close read end

        char msg[] = "Hello Child";

        write(fd[1], msg, strlen(msg)+1);

    } else {
        // Child process

        close(fd[1]); // close write end

        char buffer[100];

        read(fd[0], buffer, sizeof(buffer));

        printf("Child received: %s\n", buffer);
    }

    return 0;
}
EOF

gcc code.c -o code
./code

Child received: Hello Child


In [3]:
%%bash
cat <<EOF > code.c
#include <stdio.h>
#include <unistd.h>
#include <stdlib.h>

int main() {

    int fd[2];
    pipe(fd);

    int id = fork();

    if(id == 0) {
        // Child

        close(fd[0]);

        int x = 6;

        write(fd[1], &x, sizeof(x));

        close(fd[1]);
    }

    else {
        // Parent

        close(fd[1]);

        int num;

        read(fd[0], &num, sizeof(num));

        num = num * num;

        printf("Parent squared value: %d\n", num);

        close(fd[0]);
    }

    return 0;
}
EOF

gcc code.c -o code
./code

Parent squared value: 36


In [5]:
%%bash
cat <<EOF > code.c
#include <stdio.h>
#include <unistd.h>

int main() {

    int p2c[2];  // Parent → Child
    int c2p[2];  // Child → Parent

    pipe(p2c);
    pipe(c2p);

    int id = fork();

    if(id == 0) {
        // CHILD PROCESS

        close(p2c[1]);
        close(c2p[0]);

        int num;

        read(p2c[0], &num, sizeof(num));

        printf("Child received: %d\n", num);

        num = num * 2;

        write(c2p[1], &num, sizeof(num));

        close(p2c[0]);
        close(c2p[1]);
    }

    else {
        // PARENT PROCESS

        close(p2c[0]);
        close(c2p[1]);

        int value = 10;

        write(p2c[1], &value, sizeof(value));

        read(c2p[0], &value, sizeof(value));

        printf("Parent received back: %d\n", value);

        close(p2c[1]);
        close(c2p[0]);
    }

    return 0;
}
EOF

gcc code.c -o code
./code

Child received: 10
Parent received back: 20


DRY RUN

Parent sends: 10
Child receives: 10
Child processes: 20
Child sends back: 20
Parent receives: 20

**SIGNALS**

A notification sent to a process to inform it that an event occurred.

Example events:

Ctrl + C pressed
Timer expired
Illegal memory access
Kill command

In [6]:
%%bash
cat <<EOF > code.c
#include <stdio.h>
#include <signal.h>
#include <unistd.h>

void handler(int signum) {
    printf("\nCaught signal %d (SIGINT)\n", signum);
}

int main() {

    signal(SIGINT, handler);

    while(1) {
        printf("Running...\n");
        sleep(1);
    }

    return 0;
}
EOF

gcc code.c -o code
./code

Process is interrupted.


**SIGTERM: **

In the terminal do:

ps aux | grep "./code_sigterm_example"

and then:

kill -15 PID

**SIGKILL**
In the terminal do:

ps aux | grep "./code_sigterm_example"

and then:

kill -9 PID